In [ ]:
# Full name
NAME = ""
# Institutional email (hm.edu or hmtm.de)
EMAIL = ""

<a href="https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A3_existing_models/10_1_pretrained_models_and_embeddings.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Using Large Models

+ **AI in Culture and Arts - Tech Crash Course**
+ **Date:** 17.06.2026
+ **Author:** Dr. Benedikt Zönnchen

In the following we will create music sheets and sound. For those tasks ``Python`` requires external programs that you should install if you are working locally:

1. [Musescore](https://musescore.org/de) (for generating sheets)
2. [FluidSynth](https://www.fluidsynth.org/) (for generating sound)

In [ ]:
#@title install dependencies to play sound
%%capture
print('installing fluidsynth...')
!apt-get install fluidsynth > /dev/null
!cp /usr/share/sounds/sf2/FluidR3_GM.sf2 ./font.sf2
print('done!')

In [ ]:
#@title install dependencies to show score in music notation
%%capture
print('installing musescore3...')
!apt-get install musescore3 > /dev/null
print('done!')

In [ ]:
#@title Setup: install required Python packages

%pip install music21
%pip install pyfluidsynth

%pip install matplotlib
%pip install seaborn
%pip install scikit-learn

%pip install pandas
%pip install numpy
%pip install torch

%pip install otter-grader==5.5.0

In [ ]:
#@title Setup: download assignment files (run this cell)

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # download test files
    import requests, os

    folders = ['tests', 'data', 'models']
    link = "https://api.github.com/repos/aica-wavelab/aica-assignments/contents/A3_existing_models"

    def download(entry, dest):
        if entry.get('type') != 'file' or not entry.get('download_url'):
            return
        r = requests.get(entry['download_url'])
        r.raise_for_status()
        with open(dest, 'wb') as out:
            out.write(r.content)

    for folder in folders:
        os.makedirs(folder, exist_ok=True)
        for f in requests.get(f"{link}/{folder}").json():
            download(f, f"{folder}/{f['name']}")

    for f in requests.get(link).json():
        if f['name'].endswith('.py'):
            download(f, f['name'])

    # Initialize Otter
    import otter
    grader = otter.Notebook(colab=True)
else:
    import otter
    grader = otter.Notebook('10_1_pretrained_models_and_embeddings.ipynb')

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

## 26 Pre-trained Models and What They've Learned

In the previous notebooks we built and trained models — a Markov chain, an LSTM, and a Transformer — entirely from scratch on ~1000 German folk song melodies. In practice, training from scratch on a small dataset has a fundamental limitation: the model can only learn from what it sees during training, and a small dataset provides little signal.

A different approach is to start from a **pre-trained model**: a model that has *already* been trained, possibly on a much larger dataset or for a much longer time. We then use this model as a starting point rather than reinventing the wheel.

In this notebook we explore what our pre-trained transformer has *learned* by looking inside it. Concretely, we will examine its **embeddings**, that is, the dense numerical representations the model assigns to every token and every melody.

### What is an embedding?

Every token (e.g. ``60`` for MIDI pitch 60, or ``_`` for a held note) is represented inside the model as a **vector** which is just a list of numbers. These vectors are called *embeddings* and they live in an $n$-dimensional space (in our model, $n = 32$).

The key insight is that these vectors are **learned**: during training, the model adjusted them so that tokens that appear in similar musical contexts end up close together in the embedding space.

This means we can treat the embedding space as a kind of *map* of the model's musical knowledge:
- notes that sound similar or serve similar musical roles will be nearby (according to the training data)
- the hold symbol ``_`` will be in a different region from pitch tokens
- relationships like "these notes belong to the same scale" may be visible as clusters

### 26.1 Loading the Pre-trained Transformer

We start by reconstructing the exact architecture that was used to train the model and then loading the saved weights.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile, glob
import music21 as m21

from encoder import PianoRollEncoder, StringToIntEncoder, TERM_SYMBOL
from files import load_midi_files
from transformer import TransformerDecoder

In [ ]:
# Configure our plotting engine to get nice visualiziations
sns.set_theme(style="whitegrid")
sns.set_context("talk", font_scale=0.8)
sns.set_palette("viridis")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
# Recreate the exact same vocabulary the model was trained on
with zipfile.ZipFile('data/deu_folk_songs.zip', 'r') as z:
    z.extractall('data/deu_folk_songs/')

time_step = 0.5
mid_files = glob.glob('data/deu_folk_songs/**/*.mid', recursive=True)
streams   = load_midi_files(mid_files, time_step=time_step, transpose_to_major=True, max_files=1000)

piano_roll_encoder = PianoRollEncoder(time_step=time_step)
piano_rolls, _     = piano_roll_encoder.encode_streams(streams)
string_to_int      = StringToIntEncoder(piano_rolls)
vocab_size         = len(string_to_int)

print(f'Vocabulary size: {vocab_size}')
print(f'Tokens: {string_to_int.itos}')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if not torch.cuda.is_available():
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu') # for mac gpu
sequence_len = 64
n_embd       = 32
n_heads      = 4
n_blocks     = 2
dropout      = 0.0   # no dropout at inference time

model = TransformerDecoder(
    vocab_size   = vocab_size,
    sequence_len = sequence_len,
    n_embd       = n_embd,
    n_heads      = n_heads,
    n_blocks     = n_blocks,
    dropout      = dropout,
).to(device)

model.load_state_dict(torch.load('models/transformer_model_1000_120.pt', map_location=device))
model.eval()
print('Model loaded successfully.')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

### 26.2 Token Embeddings

The very first thing the transformer does with any input token is look it up in an **embedding table**: a matrix of shape `(vocab_size, n_embd)`. Each row in this table is the *embedding vector* of one token.

We can extract this matrix directly and visualise it.

In [ ]:
# Extract the token embedding matrix from the trained model
token_embs = model.token_embedding_table.weight.detach().cpu().numpy()  # (vocab_size, n_embd)

print(f'Embedding matrix shape: {token_embs.shape}')
print(f'{vocab_size} tokens, each represented as a {n_embd}-dimensional vector')

32 dimensions are impossible to visualise directly. We use **t-SNE** (t-Distributed Stochastic Neighbour Embedding) to project the embeddings down to 2D while trying to preserve the neighbourhood structure: tokens that are close in 32D will tend to stay close in 2D.

In [ ]:
from sklearn.manifold import TSNE

# t-SNE with perplexity=5 works well for a small vocabulary
tsne   = TSNE(n_components=2, perplexity=5, random_state=42, max_iter=2000)
coords = tsne.fit_transform(token_embs)   # (vocab_size, 2)

# Build a readable label for each token
labels = []
for i in range(vocab_size):
    sym = string_to_int.decode(i)
    if sym == TERM_SYMBOL:
        labels.append('Start/End')
    elif sym == '_':
        labels.append('hold')
    elif sym == 'r':
        labels.append('rest')
    else:
        note = m21.note.Note(int(sym))
        labels.append(f'{note.nameWithOctave}\n({sym})')

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(coords[:, 0], coords[:, 1], s=80, alpha=0.8)
for i, label in enumerate(labels):
    ax.annotate(label, coords[i], fontsize=10, ha='center', va='bottom')
ax.set_title('t-SNE of token embeddings (pre-trained transformer)')
ax.set_xlabel('t-SNE dimension 1')
ax.set_ylabel('t-SNE dimension 2')
plt.tight_layout()
plt.show()

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 26.1:** Look at the t-SNE visualisation above.

1. Do you see any clusters? Which tokens appear close to each other?
2. Where are the special symbols ``hold``, ``rest`` and ``END`` in the plot? Does their position make musical sense to you?
3. The t-SNE layout changes with the random seed. What does this tell you about the limits of t-SNE as an analysis tool?

---

*Your answer here.*

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### 26.3 Measuring Similarity Between Tokens

To quantify how similar two token embeddings are we use **cosine similarity**:

$$\text{sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \cdot \|\mathbf{b}\|}$$

A value of $1$ means the two vectors point in the same direction (maximally similar); $0$ means they are orthogonal (unrelated); $-1$ means they point in opposite directions.

In [ ]:
def cosine_similarity(a, b):
    """Cosine similarity between two 1-D numpy arrays."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Compare C4 (MIDI 60) with neighbouring notes
token_c4  = string_to_int.encode('60')
token_d4  = string_to_int.encode('62')
token_c5  = string_to_int.encode('72')
token_hld = string_to_int.encode('_')

print(f'sim(C4, D4)   = {cosine_similarity(token_embs[token_c4], token_embs[token_d4]):.3f}')
print(f'sim(C4, C5)   = {cosine_similarity(token_embs[token_c4], token_embs[token_c5]):.3f}')
print(f'sim(C4, hold) = {cosine_similarity(token_embs[token_c4], token_embs[token_hld]):.3f}')

---

🖍 **Exercise 26.2:** Compute the cosine similarity between two pairs of tokens of your choice. Store the results in variables ``sim_pair_1`` and ``sim_pair_2``.

🗣 **Hint:** Use the ``cosine_similarity`` function defined above and look up token indices with ``string_to_int.encode('token_string')``.

---

In [ ]:
# Example: E4 (64) vs G4 (67)  — both are common notes in C major
token_e4 = ...
token_g4 = ...
sim_pair_1 = ...

# Example: rest vs hold — both are non-pitch events
token_rest = ...
sim_pair_2 = ...

print(f'sim(E4, G4)   = {sim_pair_1:.3f}')
print(f'sim(hold, rest) = {sim_pair_2:.3f}')

In [ ]:
grader.check("q262")

### 26.4 Melody Embeddings

So far we have been looking at *token* embeddings — representations of individual notes. But we can also represent an entire *melody* as a single vector.

The idea: run a melody through all the transformer's blocks and then **average the output vectors over all time steps**. The result is a single vector that captures the "character" of the melody as seen through the eyes of the pre-trained model.

This is called **mean pooling** and it is one of the simplest and most robust ways to obtain a fixed-size representation from a variable-length sequence.

In [ ]:
def get_melody_embedding(model, melody: list, string_to_int, device) -> np.ndarray:
    """
    Encode a melody (list of token strings) into a single n_embd-dimensional vector
    by running it through the transformer and mean-pooling the final hidden states.
    """
    seq_len = model.sequence_len
    padded  = [TERM_SYMBOL] * seq_len + melody
    tokens  = string_to_int.encode_sequence(padded)[-seq_len:]

    idx = torch.tensor([tokens], dtype=torch.long, device=device)
    with torch.no_grad():
        hidden = model.encode(idx)   # (1, T, n_embd)
    return hidden[0].mean(dim=0).cpu().numpy()   # (n_embd,)

In [ ]:
# Encode all folk-song melodies into embedding vectors
piano_rolls_int = string_to_int.encode_sequences(piano_rolls)

melody_vecs = []
for pr in piano_rolls[:80]:   # use first 80 for speed
    melody_vecs.append(get_melody_embedding(model, pr, string_to_int, device))

melody_vecs = np.array(melody_vecs)   # (80, n_embd)
print(f'Melody embedding matrix: {melody_vecs.shape}')

In [ ]:
# Visualise melody embeddings in 2D
tsne_m = TSNE(n_components=2, perplexity=10, random_state=0, max_iter=2000)
coords_m = tsne_m.fit_transform(melody_vecs)

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(coords_m[:, 0], coords_m[:, 1], s=60, alpha=0.7)
for i in range(len(coords_m)):
    ax.annotate(str(i), coords_m[i], fontsize=6, ha='center', va='bottom')
ax.set_title('t-SNE of melody embeddings (80 German folk songs)')
ax.set_xlabel('t-SNE dimension 1')
ax.set_ylabel('t-SNE dimension 2')
plt.tight_layout()
plt.show()

### 26.5 Finding Similar Melodies

Now that every melody is a vector, we can find the **nearest neighbours** of any query melody — melodies that the model considers similar. This opens up a creative tool: given a melody you like, retrieve the most similar ones from a collection.

In [ ]:
def find_nearest(query_idx, melody_vecs, top_n=3, reverse=False):
    """
    Return the top_n melody indices most similar to query_idx (by cosine similarity).
    If reverse==True, it returns the top_n indices most different to query_idx.
    """
    query = melody_vecs[query_idx]
    sims  = melody_vecs @ query / (
        np.linalg.norm(melody_vecs, axis=1) * np.linalg.norm(query) + 1e-9
    )
    # exclude the query itself
    sims[query_idx] = -1.0
    ranked = np.argsort(sims)[::-1]
    if reverse:
        return ranked[-top_n:], sims[ranked[-top_n:]]
    else:
        return ranked[:top_n], sims[ranked[:top_n]]

In [ ]:
query = 0
neighbours, scores = find_nearest(query, melody_vecs)
print(f'Melody {query} is most similar to:')
for idx, score in zip(neighbours, scores):
    print(f'  Melody {idx:3d}  (cosine similarity = {score:.3f})')

In [ ]:
# Listen to the query and its nearest neighbour
score_q = piano_roll_encoder.decode_stream(piano_rolls[query])
print('Query melody:')
score_q.show('midi')

In [ ]:
score_n = piano_roll_encoder.decode_stream(piano_rolls[neighbours[0]])
print(f'Nearest neighbour (melody {neighbours[0]}):')
score_n.show('midi')

<!-- BEGIN QUESTION -->

### 26.6 Finding Different Melodies

Given a melody we can also find one that is quite different.

In [ ]:
query = 0
neighbours, scores = find_nearest(query, melody_vecs, reverse=True)
print(f'Melody {query} is most different to:')
for idx, score in zip(neighbours, scores):
    print(f'  Melody {idx:3d}  (cosine similarity = {score:.3f})')

In [ ]:
# Listen to the query and its furthest neighbour
score_q = piano_roll_encoder.decode_stream(piano_rolls[query])
print('Query melody:')
score_q.show('midi')

In [ ]:
score_n = piano_roll_encoder.decode_stream(piano_rolls[neighbours[0]])
print(f'Nearest neighbour (melody {neighbours[0]}):')
score_n.show('midi')

---

🖍 **Exercise 26.3:** Listen to query melody ``0`` and its nearest neighbour. 

1. Do they sound similar to you? In what way?
2. Do you think a human musician would judge the same pair as most similar? What criteria might a human use that the model embedding might miss?
3. Compare this with the melody the model seems to evaluate as most different. Do you agree?

---

*Your answer here.*

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### 26.6 Summary

In this notebook we opened the black box of a pre-trained transformer and examined what it has learned:

| Concept | What we did |
|---------|-------------|
| Token embeddings | Extracted the embedding table and visualised it with t-SNE |
| Cosine similarity | Measured how close two token representations are |
| Melody embeddings | Mean-pooled hidden states to get a per-melody vector |
| Nearest neighbours | Used embedding similarity as a musical retrieval criterion |

In the next notebook 

<a href="https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A3_existing_models/10_2_decoding_strategies.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a> 

we will look at the *output* end of the model: how exactly does the model turn a probability distribution into a generated note, and what creative choices do we have there?

---

# Bonus: Embeddings for Language

The following bonus code shows how to use a transformer model to compare the semantic meaning of sentences / paragraphs. We use the `paraphrase-multilingual-MiniLM-L12-v2` model which is a so-called `SentenceTransformer`, that is, a transformer trained not to predict the next word of a sentence but to compare text. One difference is that during training the transformer is allowed to look at the whole text (into the future). It is also **not** a decoder only transformer.

In [ ]:
#@title Setup: install required Python packages

%pip install sentence-transformers
%pip install scikit-learn

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load the multilingubbal transformer model (this can take some time)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

In [ ]:
# 2. Define sentences to compare
sentences = [
    "The weather is beautiful today.",      # English Base
    "It is a lovely sunny day outside.",    # English Similar Meaning
    "Das Wetter ist heute wunderschön.",    # German Translation (Cross-lingual)
    "I need to fix my car's engine."        # English Completely Unrelated
]

# 3. Convert sentences into numerical vectors (embeddings)
embeddings = model.encode(sentences)

# 4. Calculate Cosine Similarity between the base sentence and the others
# We compare the first sentence (index 0) against all sentences
similarities = cosine_similarity([embeddings[0]], embeddings)[0]

# 5. Print results
print(f"Base Sentence: '{sentences[0]}'\n")
for i, score in enumerate(similarities):
    print(f"Compared to: '{sentences[i]}'")
    print(f"Cosine Similarity Score: {score:.4f}\n")

We can also compare single words:

In [ ]:
words = ['King', 'Queen', 'Man', 'Woman', 'Apple']
embeddings = model.encode(words)

In [ ]:
similarities = cosine_similarity([embeddings[0]], embeddings)[0] # King compared to all others
similarities

In [ ]:
cosine_similarity([embeddings[0]-embeddings[2]], [embeddings[1]-embeddings[3]]) # King - Man compared to Queen - Woman

In [ ]:
cosine_similarity([embeddings[0]-embeddings[2]], [embeddings[1]-embeddings[4]]) # King - Man compared to Queen - Apple